# Advanced Problems with Solutions: Decimal Performance Considerations

Topic focus: memory footprint, object creation cost, arithmetic speed, benchmarking pitfalls, precision trade-offs, and when `Decimal` is worth the performance cost.

In [1]:
import sys
import time
import math
import statistics
from decimal import Decimal, localcontext

## Problem 1: Measure Object Memory Footprint

Compare the memory footprint of these objects:

```python
3.1415
Decimal('3.1415')
Decimal('3.1415926535897932384626433832795028841971')
```

Explain why `Decimal` objects usually take more memory than floats.

In [2]:
values = [
    3.1415,
    Decimal('3.1415'),
    Decimal('3.1415926535897932384626433832795028841971')
]

for value in values:
    print(repr(value))
    print('type:', type(value).__name__)
    print('size:', sys.getsizeof(value), 'bytes')
    print('-' * 50)

3.1415
type: float
size: 24 bytes
--------------------------------------------------
Decimal('3.1415')
type: Decimal
size: 120 bytes
--------------------------------------------------
Decimal('3.1415926535897932384626433832795028841971')
type: Decimal
size: 120 bytes
--------------------------------------------------


### Solution

A float has a fixed-size binary representation.

A `Decimal` stores more information, including sign, digits, exponent, and metadata needed for decimal arithmetic.

Therefore, a `Decimal` object usually has a much larger memory footprint than a float.

## Problem 2: Benchmark Float Creation vs Decimal Creation

Benchmark the cost of repeatedly creating:

```python
3.1415
Decimal('3.1415')
Decimal(3.1415)
```

Which one is slowest, and why?

In [3]:
def time_function(func, n=1_000_000):
    start = time.perf_counter()
    func(n)
    end = time.perf_counter()
    return end - start


def create_float(n):
    for _ in range(n):
        x = 3.1415


def create_decimal_from_string(n):
    for _ in range(n):
        x = Decimal('3.1415')


def create_decimal_from_float(n):
    for _ in range(n):
        x = Decimal(3.1415)


n = 1_000_000

print('float literal          :', time_function(create_float, n))
print('Decimal from string    :', time_function(create_decimal_from_string, n))
print('Decimal from float     :', time_function(create_decimal_from_float, n))

float literal          : 0.03278739936649799
Decimal from string    : 0.46183190029114485
Decimal from float     : 1.148450800217688


### Solution

Creating a float literal is very fast.

Creating a `Decimal` from a string is slower because Python must parse the string into a decimal representation.

`Decimal(3.1415)` can also be expensive and is usually not recommended because it imports the exact binary float approximation into the Decimal object.

## Problem 3: Avoid Repeated Decimal Construction

A developer writes this loop:

```python
for i in range(n):
    total += Decimal('0.01')
```

Rewrite it to avoid unnecessary object construction and benchmark both versions.

In [4]:
def repeated_construction(n):
    total = Decimal('0')
    for _ in range(n):
        total += Decimal('0.01')
    return total


def reused_decimal(n):
    total = Decimal('0')
    cent = Decimal('0.01')
    for _ in range(n):
        total += cent
    return total


n = 500_000

start = time.perf_counter()
result_1 = repeated_construction(n)
time_1 = time.perf_counter() - start

start = time.perf_counter()
result_2 = reused_decimal(n)
time_2 = time.perf_counter() - start

print('repeated construction:', time_1, result_1)
print('reused Decimal      :', time_2, result_2)
print('same result         :', result_1 == result_2)
print('speedup             :', time_1 / time_2)

repeated construction: 0.31694169994443655 5000.00
reused Decimal      : 0.05778499972075224 5000.00
same result         : True
speedup             : 5.484843843143842


### Solution

The optimized version creates `Decimal('0.01')` once and reuses it.

This is faster because constructing Decimal objects is relatively expensive compared with reusing an existing object.

## Problem 4: Compare Addition Performance

Benchmark repeated addition using floats and Decimals.

Use pre-created operands so that you measure arithmetic cost, not construction cost.

In [5]:
def float_addition(n):
    a = 3.1415
    b = 2.7182
    total = 0.0
    for _ in range(n):
        total += a + b
    return total


def decimal_addition(n):
    a = Decimal('3.1415')
    b = Decimal('2.7182')
    total = Decimal('0')
    for _ in range(n):
        total += a + b
    return total


n = 1_000_000

start = time.perf_counter()
float_result = float_addition(n)
float_time = time.perf_counter() - start

start = time.perf_counter()
decimal_result = decimal_addition(n)
decimal_time = time.perf_counter() - start

print('float time  :', float_time)
print('decimal time:', decimal_time)
print('ratio       :', decimal_time / float_time)

float time  : 0.06408809963613749
decimal time: 0.266126099973917
ratio       : 4.152504154201132


### Solution

Decimal addition is usually slower than float addition.

Floats are implemented using native binary floating-point operations, which CPUs optimize heavily.

Decimals require base-10 arithmetic with arbitrary precision behavior, context handling, and more complex internal representation.

## Problem 5: Compare Square Root Performance

Benchmark:

```python
math.sqrt(Decimal_value_as_float)
Decimal.sqrt()
```

Explain the performance and precision trade-off.

In [6]:
def float_sqrt(n):
    a = 3.1415
    total = 0.0
    for _ in range(n):
        total += math.sqrt(a)
    return total


def decimal_sqrt(n):
    a = Decimal('3.1415')
    total = Decimal('0')
    for _ in range(n):
        total += a.sqrt()
    return total


n = 100_000

start = time.perf_counter()
float_result = float_sqrt(n)
float_time = time.perf_counter() - start

start = time.perf_counter()
decimal_result = decimal_sqrt(n)
decimal_time = time.perf_counter() - start

print('float sqrt time  :', float_time)
print('decimal sqrt time:', decimal_time)
print('ratio            :', decimal_time / float_time)

float sqrt time  : 0.014223899692296982
decimal sqrt time: 0.2562268003821373
ratio            : 18.01382222351428


### Solution

`math.sqrt()` is fast because it uses binary floating-point arithmetic.

`Decimal.sqrt()` is slower because it respects Decimal precision and context.

Use `Decimal.sqrt()` when decimal precision matters. Use `math.sqrt()` when float precision is sufficient and speed matters more.

## Problem 6: Precision vs Performance

Benchmark `Decimal.sqrt()` for the same number using precision values:

```python
10, 28, 50, 100
```

What happens as precision increases?

In [7]:
def sqrt_with_precision(precision, n=20_000):
    with localcontext() as ctx:
        ctx.prec = precision
        x = Decimal('2')
        start = time.perf_counter()
        for _ in range(n):
            y = x.sqrt()
        end = time.perf_counter()
        return end - start, y


for precision in [10, 28, 50, 100]:
    elapsed, result = sqrt_with_precision(precision)
    print('precision:', precision)
    print('time     :', elapsed)
    print('sqrt(2)  :', result)
    print('-' * 60)

precision: 10
time     : 0.02703540027141571
sqrt(2)  : 1.414213562
------------------------------------------------------------
precision: 28
time     : 0.0664793998003006
sqrt(2)  : 1.414213562373095048801688724
------------------------------------------------------------
precision: 50
time     : 0.06910210009664297
sqrt(2)  : 1.4142135623730950488016887242096980785696718753769
------------------------------------------------------------
precision: 100
time     : 0.16507829912006855
sqrt(2)  : 1.414213562373095048801688724209698078569671875376948073176679737990732478462107038850387534327641573
------------------------------------------------------------


### Solution

Higher precision usually increases computation time.

This is because Decimal operations must process more significant digits.

The trade-off is straightforward: more precision gives more accurate decimal results, but costs more CPU time.

## Problem 7: Build a Simple Benchmark Helper

Single benchmark runs can be noisy.

Write a helper function that runs a benchmark multiple times and reports:

- minimum time
- average time
- median time

Then use it to compare float and Decimal multiplication.

In [8]:
def benchmark(func, repeats=5):
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        func()
        end = time.perf_counter()
        timings.append(end - start)
    return {
        'min': min(timings),
        'mean': statistics.mean(timings),
        'median': statistics.median(timings)
    }


def float_multiplication():
    a = 1.2345
    b = 9.8765
    total = 0.0
    for _ in range(500_000):
        total += a * b
    return total


def decimal_multiplication():
    a = Decimal('1.2345')
    b = Decimal('9.8765')
    total = Decimal('0')
    for _ in range(500_000):
        total += a * b
    return total


print('float  :', benchmark(float_multiplication))
print('Decimal:', benchmark(decimal_multiplication))

float  : {'min': 0.019620000384747982, 'mean': 0.026320300064980984, 'median': 0.022180099971592426}
Decimal: {'min': 0.0918453000485897, 'mean': 0.10303820017725229, 'median': 0.10582230053842068}


### Solution

Benchmarking once can be misleading because of background processes, CPU scheduling, caching, and interpreter overhead.

Running several trials and comparing minimum, mean, and median times gives a more reliable picture.

## Problem 8: Estimate List Memory Usage

Create a list of `100_000` floats and a list of `100_000` Decimals.

Estimate total memory usage using:

```python
sys.getsizeof(list_object) + sum(sys.getsizeof(x) for x in list_object)
```

Compare the results.

In [9]:
n = 100_000

float_list = [3.1415 for _ in range(n)]
decimal_list = [Decimal('3.1415') for _ in range(n)]

float_memory = sys.getsizeof(float_list) + sum(sys.getsizeof(x) for x in float_list)
decimal_memory = sys.getsizeof(decimal_list) + sum(sys.getsizeof(x) for x in decimal_list)

print('float list memory  :', float_memory, 'bytes')
print('Decimal list memory:', decimal_memory, 'bytes')
print('ratio              :', decimal_memory / float_memory)

float list memory  : 3200984 bytes
Decimal list memory: 12800984 bytes
ratio              : 3.9990777835815488


### Solution

A list stores references to objects, so the list itself has overhead.

The objects inside the list also consume memory.

Since each Decimal object is larger than each float object, large collections of Decimals can require much more memory.

## Problem 9: Choose the Right Numeric Type

For each scenario, choose `float` or `Decimal` and justify your answer:

1. Rendering physics particles in a game engine.
2. Calculating tax on an invoice.
3. Running millions of approximate simulations.
4. Splitting money exactly among users.
5. Computing graphics shader values on a GPU.

### Solution

1. Game physics: usually `float`, because speed matters and small binary rounding errors are acceptable.
2. Tax calculation: usually `Decimal`, because exact decimal rounding rules matter.
3. Approximate simulations: usually `float`, because performance matters more than exact decimal representation.
4. Money splitting: usually `Decimal`, because cents must be handled correctly.
5. GPU shader values: usually `float`, because GPUs are optimized for floating-point arithmetic, not Decimal arithmetic.

Best practice: use `Decimal` when decimal correctness matters more than speed. Use `float` when performance and approximate numeric behavior are acceptable.

## Problem 10: Optimize a Decimal-Based Invoice Calculation

The following function is correct but inefficient:

```python
def invoice_total(prices, tax_rate):
    total = Decimal('0')
    for price in prices:
        total += Decimal(str(price))
    return total + total * Decimal(str(tax_rate))
```

Rewrite it so repeated conversions are reduced and rounding to cents is explicit.

In [10]:
def invoice_total_optimized(prices, tax_rate):
    cent = Decimal('0.01')
    tax_rate = Decimal(str(tax_rate))
    decimal_prices = [Decimal(str(price)) for price in prices]

    subtotal = sum(decimal_prices, Decimal('0'))
    tax = (subtotal * tax_rate).quantize(cent)
    total = (subtotal + tax).quantize(cent)

    return subtotal, tax, total


prices = ['19.99', '5.49', '12.30', '100.00']
tax_rate = '0.0825'

subtotal, tax, total = invoice_total_optimized(prices, tax_rate)

print('subtotal:', subtotal)
print('tax     :', tax)
print('total   :', total)

subtotal: 137.78
tax     : 11.37
total   : 149.15


### Solution

The optimized version follows several best practices:

- Convert inputs once.
- Use `Decimal(str(value))` for safer conversion.
- Use `sum(..., Decimal('0'))` to keep the calculation Decimal-based.
- Use `quantize(Decimal('0.01'))` to make cent-level rounding explicit.
- Avoid unnecessary Decimal construction inside tight loops.